## Import libraries

In [19]:
import os
import json
import pandas as pd
import pickle
import numpy as np
import copy

## Load the wind data

In [20]:
wnd_records = {}

for record in os.listdir(f"Results\\stick_model") : 
    f_name,ext = os.path.splitext(record)
    if ext == '.pickle' :          
        with open(f"Results\\stick_model\\{record}","rb") as f:
            data = pickle.load(f)
        wnd_records[f_name] = data
print(wnd_records.keys())

dict_keys(['unscaled_TS_112_0'])


## Minor edits to floor labels to match OpenSees floor labels

In [21]:
new_floor_keys = [72,73, 74, 75, 76, 77, 78, 79, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720]
for file in wnd_records.keys() : 
    initial_floor_keys = [x for x in wnd_records[file] if x not in ['metadata',]]
    # Note : initial_floor_keys are negative, because they come from virtual centroid nodes,
    # which are ordered such as : -1 is storey 1, -19 is storey 19. 
    for key, new_key in zip(initial_floor_keys,new_floor_keys):
        wnd_records[file][new_key] = wnd_records[file].pop(key)

## Initiate a new JSON file containing wind forces

In [22]:
enabled = ['Fx'] # Indicate which of the directions should be enabled in the assessment
options = ['Fx','Fy','Tz']

selected= {}
for o in options : 
    if o in enabled : 
        selected[o] = True
    else : 
        selected[o] = False
print(selected)

{'Fx': True, 'Fy': False, 'Tz': False}


In [23]:
proxy_wnd_tunnel = True
if proxy_wnd_tunnel == True :
    ms = 396.24 # corresponds to H_full/H_model (i.e. 79.248 m / 0.2 m)
    Building_depth_proxy = 1800/396.24 # 1800 inches / ms
    Building_breadth_proxy = 1800/ 396.24 # 1800 inches / ms

In [24]:
proxy_wnd_tunnel = True

for file in wnd_records :     
    base_json = {}
    Fx = []
    Fy = []
    Tz = []

    if proxy_wnd_tunnel == True : 
        base_json['D'] = Building_depth_proxy
        base_json['B'] = Building_breadth_proxy
    else : 
        base_json['D'] = wnd_records[file]['metadata']['D']*1000/25.4 # conversion meters to inches
        base_json['B'] = wnd_records[file]['metadata']['B']*1000/25.4 # conversion meters to inches        
        
    base_json['H'] = wnd_records[file]['metadata']['H']*1000/25.4 # conversion meters to inches
    base_json['fs'] = float(wnd_records[file]['metadata']['fs'])
    base_json['Vref'] = wnd_records[file]['metadata']['Vref']*1000/25.4 # conversion : meter/s to in/sec

    
    for floor in wnd_records[file].keys(): # will count from floor 1 to floor 19 :
        if floor != 'metadata' : 
            #print(floor)  
            # convert Newtons to kilonewtons + to kip or kip-inch
            # Also : as a simplification for 2D, assume both frames are loaded equally (thus, halving the wind load)
            Fx.append((wnd_records[file][floor][:,0]/1000*0.22480894387096/2).tolist())
            Fy.append((wnd_records[file][floor][:,1]/1000*0.22480894387096/2).tolist())
            Tz.append((wnd_records[file][floor][:,2]/1000*0.22480894387096*1000/25.4/2).tolist())    

    for direction in selected : 
        if (direction == 'Fx') and (selected[direction] == True) :
            base_json[direction] = Fx
            continue            
        if (direction == 'Fy') and (selected[direction] == True): 
            base_json[direction] = Fy
            continue            
        if (direction == 'Tz') and (selected[direction] == True) : 
            base_json[direction] = Tz
            continue
        else : 
            base_json[direction] = np.array([]).reshape([0,0]).tolist()
                    
    base_json['t'] = [wnd_records[file]['metadata']['t'].tolist()]    
    
    
    ref_str = '-'.join(enabled)

    # Save to json format, in according directory (with some nice formating) :
    save_dir = os.path.join(os.path.abspath(''),'Results','stick_model')
    if not os.path.isdir(save_dir) : 
        os.makedirs(save_dir)

    with open(f"Results\\stick_model\\{file}-{ref_str}.json","w",encoding='utf-8') as f:
        json.dump(base_json,f, ensure_ascii=False, indent=4)
 